# Batch Embedding Multilingual News Corpus on Kaggle GPU (T4)
This notebook runs offline batch embedding for English, Hindi, Bengali, and Tamil news articles using **BGE-M3 (fp16)**.
The generated vectors and chunk metadata are saved for Weaviate vector indexing.

In [ ]:
# Install required dependencies
!pip install -q FlagEmbedding transformers torch pandas pyarrow tqdm

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from FlagEmbedding import BGEM3FlagModel

print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

In [ ]:
# Load chunked corpus
CHUNKS_PATH = "chunks.parquet"
OUTPUT_VECTORS = "vectors.npy"

df = pd.read_parquet(CHUNKS_PATH)
texts = df["text"].fillna("").astype(str).tolist()
print(f"Total chunks to embed: {len(texts)}")

# Initialize BGE-M3 in fp16
model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True, device="cuda" if torch.cuda.is_available() else "cpu")

# Generate dense embeddings in batches
batch_size = 64
output = model.encode(
    texts,
    batch_size=batch_size,
    max_length=8192,
    return_dense=True,
    return_sparse=False,
    return_colbert_vecs=False,
)

dense_vectors = np.array(output["dense_vecs"], dtype=np.float32)
print("Embeddings shape:", dense_vectors.shape)

# Save to disk
np.save(OUTPUT_VECTORS, dense_vectors)
print(f"Successfully saved {len(dense_vectors)} vectors to {OUTPUT_VECTORS}")